###### VeRL 利用装饰器将复杂的分布式通信（如数据分片、聚合、广播）抽象为简单的函数调用，实现了“声明式分布式编程”。


- 分布式调度装饰器
1. @register（核心装饰器）
- 二、性能分析装饰器
2. GPUMemoryLogger（类装饰器）
- 三、工具装饰器
4. @timeout_limit
- 四、Engine 注册装饰器
6. @register（Engine Registry）
- 五、辅助/内部装饰器
7. make_nd_compute_dataproto_dispatch_fn

# 分布式调度装饰器

1. @register（核心装饰器）
- 位置：verl/single_controller/base/decorator.py

这是 verl 框架最重要的装饰器，<font color='red'>是 HybridFlow 架构中控制流与计算流分离的关键基础设施。</font>它通过声明式编程，让开发者像写单机代码一样编写分布式逻辑。

###### 它的主要作用是将普通的 Python 方法注册为可分布式执行的方法。

当你给一个 Worker 类的方法加上 @register 时，VeRL 框架会在运行时自动拦截该方法的调用，并根据你配置的参数，自动处理数据的分发（Dispatch）和结果的收集（Collect）


## 核心参数与功能：

- <font color='red'>dispatch_mode (分发模式)</font>：定义数据如何从控制器（Controller/Driver）发送到工作节点（Worker）。
    - Dispatch.ONE_TO_ALL：广播模式。将输入数据复制并发送给所有 Worker。常用于模型权重的初始化或广播配置。
    - Dispatch.ALL_TO_ALL：散列模式。将输入数据切分（Shard），每个 Worker 接收一部分。常用于数据并行训练中的批次数据分发。
    - Dispatch.DP_COMPUTE / DP_COMPUTE_PROTO：专门针对数据并行（Data Parallel）计算优化的模式，通常涉及复杂的张量切分和协议处理。
    - Dispatch.DIRECT_ROLLOUT_METHOD：专门用于 Rollout 阶段的直接调用。
- <font color='red'>execute_mode (执行模式)</font>：定义方法在哪些节点上执行。
    - Execute.ALL：所有 Worker 都执行该方法。
    - Execute.RANK_ZERO：仅主节点（Rank 0）执行，其他节点跳过。
- <font color='red'>blocking (阻塞模式)</font>：
    - True：调用后阻塞等待所有 Worker 执行完成并返回结果。
    - False：异步执行，立即返回 Future 对象。

## 支持的 dispatch_mode

| 模式                           | 分发行为                            | 收集行为           | 典型场景                   |
| ---------------------------- | ------------------------------- | -------------- | ---------------------- |
| `ONE_TO_ALL`                 | 广播：每个 Worker 收到相同数据             | 原样返回           | 模型加载/保存、配置同步           |
| `ALL_TO_ALL`                 | 透传：数据已预分片                       | 原样返回           | 预分片的数据                 |
| `DP_COMPUTE`                 | 按列表手动切分                         | 拼接列表           | 手动数据并行                 |
| `DP_COMPUTE_PROTO`           | 自动按 `world_size` 切分 `DataProto` | 自动 `concat` 拼接 | **最常用**，数据并行训练/推理      |
| `DP_COMPUTE_PROTO_WITH_FUNC` | 切分 `DataProto` + 广播函数           | 拼接             | 动态 batch 函数            |
| `DP_COMPUTE_METRIC`          | 切分 `DataProto`                  | 不拼接，返回列表       | 指标计算                   |
| `DIRECT_ROLLOUT_METHOD`      | 直接透传                            | 直接透传           | vLLM/SGLang rollout 直通 |
| `MEGATRON_COMPUTE_PROTO`     | Megatron 3D 并行切分                | Megatron 拼接    | Megatron 后端            |
| `MEGATRON_PP_AS_DP_PROTO`    | Pipeline-as-DP 切分               | 拼接             | HybridEngine rollout   |

## 工作原理：
- @register 并不直接执行分布式逻辑，而是给被装饰函数附加一个 MAGIC_ATTR 属性（元数据字典）。在 RayWorkerGroup 初始化时，_bind_worker_method 扫描到该属性后，动态生成一个包装函数，自动完成 dispatch → execute → collect 的完整分布式调用链

In [ ]:
@register(
    dispatch_mode=Dispatch.DP_COMPUTE_PROTO,
    execute_mode=Execute.ALL,
    blocking=True,
    materialize_futures=True
)
def generate_sequences(self, prompts: DataProto):
    ...

In [ ]:
"""
它的核心哲学是：业务逻辑与分布式逻辑解耦。

开发者只需关注函数内部的计算逻辑（“做什么”），而通过装饰器参数告诉框架如何处理数据的分发和同步（“怎么做”）。

这是装饰器的工厂函数，它允许你为被装饰的方法配置“元数据”：
- dispatch_mode: 决定数据怎么分。例如是广播给所有人（ONE_TO_ALL），还是切分给不同人（ALL_TO_ALL）。
- execute_mode: 决定谁去执行。是所有 Worker 都跑一遍（ALL），还是只有主节点跑（RANK_ZERO）。
- blocking: 决定调用后是否等待结果。
- materialize_futures: 这是一个高级优化选项，用于处理异步对象（Ray Future）。


这个 register 函数通过闭包和元编程，实现了以下流程：
1.接收配置：用户指定怎么分发数据。
2.预处理数据：在执行前自动把异步 Future 转为实体数据。
3.打标签：把配置信息绑定在函数属性上，供框架后续读取和调度。
这使得 VeRL 的代码看起来非常像单机代码，但背后却拥有强大的分布式调度能力。
"""


def register(
    dispatch_mode=Dispatch.ALL_TO_ALL,
    execute_mode=Execute.ALL,
    blocking=True,
    materialize_futures=True,
):
    """Register a function with distributed execution configuration. 注册一个函数，并配置分布式执行。

    This decorator registers a function with specific dispatch and execution modes
    for distributed computation. It handles both synchronous and asynchronous
    functions, and optionally materializes futures before execution.

    Args:
        dispatch_mode:
            Dispatch mode for computation distribution. Default: Dispatch.ALL_TO_ALL.
        execute_mode:
            Execute mode for computation distribution. Default: Execute.ALL.
        blocking:
            Whether the execution should be blocking. Defaults to True.
        materialize_futures:
            Whether to materialize the data before dispatching. Defaults to True.

    Returns:
        A decorator that wraps the original function with distributed execution
        configuration.
    """
    _check_dispatch_mode(dispatch_mode=dispatch_mode)
    _check_execute_mode(execute_mode=execute_mode)

    """
    包装函数：处理异步与数据“变现”

    这里定义了真正的包装逻辑，它做了两件事：
    1. 数据“变现” (_materialize_futures)：
        - 在分布式系统中，参数可能是一个“承诺”（Future/Promise），代表一个还没计算完的结果。
        - 如果 materialize_futures=True，装饰器会在执行函数前，强制等待这些 Future 完成，把数据真正取出来（即“变现”），确保函数内部拿到的是实实在在的数据，而不是句柄。
    2. 兼容异步：
        - 代码检测了原函数是否是 async 的（inspect.iscoroutinefunction），并分别生成了同步版 inner 和异步版 async_inner 包装器，保证框架能同时支持同步和异步代码。


    """

    def decorator(func):
        @wraps(func)
        def inner(*args, **kwargs):
            if materialize_futures:
                args, kwargs = _materialize_futures(*args, **kwargs)
            return func(*args, **kwargs)

        @wraps(func)
        async def async_inner(*args, **kwargs):
            if materialize_futures:
                args, kwargs = _materialize_futures(*args, **kwargs)
            return await func(*args, **kwargs)

        """
        注入“魔法”：元数据标记

        这是整个装饰器最精髓的部分：
        - MAGIC_ATTR：这是一个特殊的字符串常量（如 "attrs_3141562937"）。
        - setattr(...)：装饰器并没有直接执行分布式逻辑，而是把你配置的参数（分发模式、执行模式等）打包成一个字典，像贴标签一样贴在了函数对象上。
        
        为什么要这么做？
        - 这是一种元编程技巧。VeRL 的框架层（如 RayWorkerGroup）在初始化时，会扫描所有 Worker 类的方法。
          一旦发现某个方法身上有这个 MAGIC_ATTR 标签，框架就知道：“哦，这个方法需要分布式处理”，
          然后框架会根据标签里的配置，动态生成相应的分发和收集代码。
        """
        wrapper = async_inner if inspect.iscoroutinefunction(func) else inner
        attrs = {
            "dispatch_mode": dispatch_mode,
            "execute_mode": execute_mode,
            "blocking": blocking,
        }
        setattr(wrapper, MAGIC_ATTR, attrs)
        return wrapper

    return decorator

# 性能分析装饰器

## GPUMemoryLogger（类装饰器）
- 位置：verl/utils/profiler/performance.py 

<font color='red'>用于在函数执行前后记录 GPU 显存占用变化。</font>

### 功能：
- <font color='red'>在函数执行前</font>记录 allocated / reserved / used / total 显存
- <font color='red'>在函数执行后</font>再次记录并输出差值
- 支持 log_only_rank_0=True 只打印 rank 0 的日志



In [ ]:
from verl.utils.profiler.performance import GPUMemoryLogger

@GPUMemoryLogger(role="actor")
def update_actor(self, batch):
    # 训练逻辑
    return

## @profile（性能分析装饰器）

- 位置：verl/utils/profiler/profile.py 

<font color='red'>用于集成 PyTorch Profiler / Nsight 等性能分析工具，对函数进行性能剖析。</font>

### 功能：
- 自动启动/停止 profiler
- 记录 CUDA kernel 执行时间、内存分配等
- 常与 GPUMemoryLogger 叠加使用（从堆栈中可见多层装饰器嵌套）


In [ ]:
# 典型调用链中可见
@profile
def compute_log_prob(self, data):
    ...

# 工具装饰器

## @timeout_limit

- 位置：verl/utils/py_functional.py 
<font color='red'>为函数执行设置超时限制，防止某些操作（如远程调用、模型加载）无限挂起。</font>

### 特点：
- use_signals=False（默认）：<font color='red'>使用 multiprocessing 子进程方式，更健壮，适用于多线程环境</font>
- use_signals=True：使用 POSIX SIGALRM 信号（已废弃，不推荐）

In [ ]:
@timeout_limit(seconds=300, use_signals=False)
def some_long_operation():
    ...

# Engine 注册装饰器

##  @register（Engine Registry）

- 位置：verl/workers/engine/base.py 

<font color='red'>这是一个独立的注册装饰器（与分布式调度的 @register 同名但不同模块），用于将自定义 Engine 注册到引擎工厂中。</font>

### 功能：
- 维护一个全局 _engines 注册表
- 通过 new(engine_name, ...) 工厂方法创建已注册引擎的实例

In [ ]:
@register
class MyCustomEngine(BaseEngine):
    ...

# 装饰器调用链示例

<font color='red'>在 verl 的实际运行中，一个 Worker 方法往往被多层装饰器嵌套包装。</font>

以 compute_log_prob 为例，从堆栈跟踪中可以看到典型的装饰顺序（从内到外）

<font color='red'>当 Driver 调用 actor_wg.compute_log_prob(data) 时，外层 @register 负责将 data 切分并分发给各 Ray Worker，Worker 端再依次经过 profile → GPUMemoryLogger → 业务逻辑，最后结果逐层收集回 Driver。</font>

In [ ]:
1. @register(dispatch_mode=DP_COMPUTE_PROTO)   ← 分布式调度
2. @profile                                     ← 性能剖析
3. @GPUMemoryLogger(role="actor")              ← 显存日志
4. 实际业务函数 compute_log_prob